In [1]:
import pandas as pd
import pickle
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

import mlflow

In [10]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='file:///C:/Users/Suchi_Kumari/mlflow/artifacts/4', creation_time=1766316594739, experiment_id='4', last_update_time=1766316594739, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [2]:
source_path = 'C://Users//Suchi_Kumari//mlflow//MLOps-1//03-Experiments//Source//yellow_tripdata_2023-03.parquet'
df = pd.read_parquet(source_path)
df.shape

(3403766, 19)

In [3]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    
    return df

In [4]:
df_train = read_dataframe(source_path)
df_train.shape

(3316216, 21)

In [5]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
# X_train = df_train[categorical + numerical]
X_train = dv.fit_transform(train_dicts)
# why not jus pass df as x_train vs using DictVectorizer

In [6]:
from sklearn.linear_model import LinearRegression

def train_model(df):
    categorical = ['PULocationID', 'DOLocationID']
    numerical = ['trip_distance']

    train_dicts = df[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer()
    X = dv.fit_transform(train_dicts)

    y = df['duration']

    model = LinearRegression()
    model.fit(X, y)

    return dv, model


In [7]:
dv, model = train_model(df_train)
print(model.intercept_)

23.90262678768884


In [ ]:
# Log model
mlflow.sklearn.log_model(
    sk_model=model,
    artifact_path="model"
)
mlflow.log_param("model_type", "LinearRegression")
mlflow.log_metric("intercept", model.intercept_)

run_id = mlflow.active_run().info.run_id
print("Run ID:", run_id)

2025/12/31 21:39:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/12/31 21:39:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run ID: 38a78f417e764ba9a96f1ba737de89e0


In [15]:
run_id = '38a78f417e764ba9a96f1ba737de89e0'

mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="nyc-taxi-model"
)

Successfully registered model 'nyc-taxi-model'.
2025/12/31 21:39:36 WARNING mlflow.tracking._model_registry.fluent: Run with id 38a78f417e764ba9a96f1ba737de89e0 has no artifacts at artifact path 'model', registering model based on models:/m-e81bf58860744925be5b472f7bc498b3 instead
2025/12/31 21:39:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-model, version 1
Created version '1' of model 'nyc-taxi-model'.


<ModelVersion: aliases=[], creation_timestamp=1767197376920, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1767197376920, metrics=None, model_id=None, name='nyc-taxi-model', params=None, run_id='38a78f417e764ba9a96f1ba737de89e0', run_link='', source='models:/m-e81bf58860744925be5b472f7bc498b3', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [17]:
import pickle
import sys

model_size_bytes = sys.getsizeof(pickle.dumps(model))
print(model_size_bytes)

4557
